# DramaWiki (wiki.d-addicts.com): Raw HTML Downloader

### Kailyn

This notebook only downloads and saves the raw HTML files. Parsing will be done in another notebook.

For each selected show (see `/dataset1_mdl/mdl_manfiest.csv`), this notebook saves three kinds of pages locally:
1. the **show page** (e.g. `wiki.d-addicts.com/Dream_High`)
2. the **OST page(s)** linked from the show page (e.g. `.../Dream_High_OST`)
3. the **actor pages for the main cast**, i.e. the people listed under *Main Cast* in the show page's *Cast* section

Everything lands in `/dataset2_mediawiki/raw_html/` (one subfolder per page type) and every request is logged in `/dataset2_mediawiki/raw_html/manifest.csv`. The summary of results is found in `/dataset2_mediawiki/drama_summary.csv`.

I'm trying using requests right now, with Selenium as a fallback. The fallback code was written with the help of genAI.

**Table of Contents**

1. [Setup](#1)
2. [Configure Parameters](#2)
3. [Setting up Caching](#3)
4. [Setting up Parsing](#4)
5. [Extract Tag Ids](#5)
6. [Collect Candidate Dramas](#6)
7. [Add Drama Metadata](#7)
8. [Run Everything](#8)
9. [Summary and Save](#9)

<a id=1></a>
## 1. Setup

In [36]:
import csv
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, unquote, urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

<a id=2></a>
## 2. Configure Parameters

In [37]:
WIKI_BASE = "https://wiki.d-addicts.com"

# Input datasets
MUSIC_CSV = "../dataset1_mdl/sample_music.csv"
CONTROL_CSV = "../dataset1_mdl/sample_control.csv"

TITLE_COLUMN = "Title_English"

# Number of control dramas to select.
# Automatically matches the number of music-centered dramas.
CONTROL_SAMPLE_SIZE = None

# Manual URL overrides for titles that do not follow the standard
# title-to-Wiki-URL convention.
# Keys should match the Title_English values in the CSVs.
WIKI_PAGE_OVERRIDES = {
    "Hospital Playlist Season 2 (2021)": "Hospital_Playlist",
    "Bravo My Life (2017)": "Bravo_My_Life_(2017)",
    "You're Too Much (2017)": "You Are Too Much",
    "Dr. Ian (2015)": "Dr._Mo_Clinic",
    "Idol in My Living Room (2022)": "There_Is_an_Idol_in_My_Living_Room",
    "Introverted Boss (2017)": "Sensitive_Boss",
    "Love (ft. Marriage and Divorce) Season 3 (2022)": "Love_(ft._Marriage_and_Divorce)",
    "Returning Student: Straight-A, but F in Love (2022)": "Returning_student:_Straight-A,_but_F_in_Love",
    "Taxi Driver (2021)": "Taxi_Driver",
    "The Good Detective Season 2 (2022)": "The_Good_Detective",
    "The Universe’s Star (2017)": "Star_of_the_Universe",
}

# Main cast
MAIN_CAST_LIMIT = None

# Politeness
DELAY_SECONDS = 2.0
MAX_ATTEMPTS = 3
BACKOFF_SECONDS = 15

USER_AGENT = "Mozilla/5.0 (course research scraper)"

# Fallback to SeleniumBase
USE_SELENIUM_FALLBACK = True

# Output folders
RAW_DIR = Path("../dataset2_mediawiki/raw_html")

for sub in ("shows", "osts", "actors"):
    (RAW_DIR / sub).mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = RAW_DIR / "manifest.csv"

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

<a id=3></a>
## 3. Fetching and saving pages

Same logic as milestone 3, although I did add a negative cache with the help of genAI because MediaWiki is missing a lot of files.

In [38]:
def polite_sleep():
    """Pause between real requests."""
    time.sleep(DELAY_SECONDS)

def looks_like_wiki_page(html):
    """True if this is a real DramaWiki content page (not a block/challenge page)."""
    return 'id="mw-content-text"' in html

_driver = None   # SeleniumBase browser, created only if the fallback is needed

def fetch_with_selenium(url):
    """Fallback: load the page in a real browser and return its HTML (or None)."""
    global _driver
    if _driver is None:
        from seleniumbase import Driver          # imported lazily: not needed otherwise
        _driver = Driver(page_load_strategy="eager")
    _driver.open(url)
    _driver.sleep(DELAY_SECONDS + 1)
    html = _driver.page_source
    return html if looks_like_wiki_page(html) else None

def close_driver():
    global _driver
    if _driver is not None:
        _driver.quit()
        _driver = None

def fetch_page(url, dest, force=False):
    """Download `url` into the file `dest` (unless already saved).
    Returns a dict describing what happened, for the manifest."""
    dest = Path(dest)
    rec = {"url": url, "path": str(dest), "status": "", "method": "", "note": "",
           "fetched_at": ""}

    # Successful local HTML cache
    if dest.exists() and not force:
        rec.update(status="cached", method="cache")
        return rec

    # Persistent negative cache
    if url in NOT_FOUND_CACHE and not force:
        rec.update(
            status="cached_404",
            method="negative_cache",
            note="Previously confirmed HTTP 404"
        )
        return rec

    html_bytes = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            resp = session.get(url, timeout=30)
        except requests.RequestException as e:
            rec["note"] = f"request error: {e}"
            polite_sleep()
            continue

        polite_sleep() # be polite!
        rec["status"] = resp.status_code

        if resp.status_code == 404:
            NOT_FOUND_CACHE.add(url)
            save_not_found_cache(NOT_FOUND_CACHE)

            rec.update(
                status=404,
                method="requests",
                note="Page does not exist; added to 404 cache"
            )
            return rec
        if resp.status_code == 200 and looks_like_wiki_page(resp.text):
            html_bytes, rec["method"] = resp.content, "requests"
            NOT_FOUND_CACHE.discard(url)
            save_not_found_cache(NOT_FOUND_CACHE)
            break

        rec["note"] = f"unexpected response (HTTP {resp.status_code}), attempt {attempt}"
        time.sleep(BACKOFF_SECONDS * attempt)

    if html_bytes is None and USE_SELENIUM_FALLBACK:
        print("  requests failed; trying the browser fallback...")
        html = fetch_with_selenium(url)
        if html:
            html_bytes, rec["method"] = html.encode("utf-8"), "selenium"
            rec["status"], rec["note"] = 200, "saved via selenium fallback"

    if html_bytes is None:
        rec["note"] = rec["note"] or "failed"
        return rec

    dest.write_bytes(html_bytes)
    rec["fetched_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    return rec

In [39]:
import json

# Persistent cache for URLs that have returned HTTP 404
NOT_FOUND_CACHE_PATH = RAW_DIR / "not_found_cache.json"

def load_not_found_cache():
    """Load previously confirmed 404 URLs."""
    if NOT_FOUND_CACHE_PATH.exists():
        with open(NOT_FOUND_CACHE_PATH, "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()


def save_not_found_cache(cache):
    """Persist confirmed 404 URLs."""
    NOT_FOUND_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

    with open(NOT_FOUND_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(sorted(cache), f, indent=2)


# Load once when the notebook runs
NOT_FOUND_CACHE = load_not_found_cache()

print(f"Loaded {len(NOT_FOUND_CACHE)} previously failed URLs.")

Loaded 0 previously failed URLs.


In [40]:
def wiki_url(page_name):
    """'Dream_High' -> 'https://wiki.d-addicts.com/Dream_High' (encoding parens etc.)."""
    return f"{WIKI_BASE}/{quote(page_name.replace(' ', '_'), safe='_')}"


def page_name_from_url(url):
    """'.../Kim_Soo_Hyun_%281988%29' -> 'Kim_Soo_Hyun_(1988)'"""
    return unquote(urlparse(url).path.lstrip("/"))


def local_path(kind, url):
    """Where a page is stored: raw_html/<kind>/<url-encoded page name>.html
    (kind is 'shows', 'osts' or 'actors')."""
    return RAW_DIR / kind / (quote(page_name_from_url(url), safe="") + ".html")

<a id=4></a>
## 4. Finding the OST and main-cast links in a saved show page

From the saved file, we need to pull the OST and cast links. I'm using beautifulsoup.

From my scan-through of the Dream High page, it looks like the following is true:
- One show page can hold **several seasons** (Dream High has Season 1 and Season 2), each with its own *Cast* section and its own *Original Soundtrack* link. I decided to look at every season.
- Under *Cast* there are labeled groups: **Main Cast**, Kirin's Students, Cameos, Extended Cast... I'm focused on Main Cast.
- Some entries have a **nested** sub-item (e.g. a child actor playing the young version of a role). I keep only the top-level entry.
- Some names are **red links** (`...redlink=1`, "page does not exist"): there's no page to download, so they're skipped.
- Some actors' page names include a **disambiguator**, like `Kim_Soo_Hyun_(1988)`.

If a season has no "Main Cast" label, I fall back to the first list. This might be changed later.

In [41]:
import re

def title_to_wiki_slug(title):
    """Convert a dataset title into a provisional DramaWiki page slug."""
    title = str(title).strip()

    # Remove a trailing release year, optionally in parentheses.
    title = re.sub(r"\s*\(?\b(?:19|20)\d{2}\)?$", "", title)

    return title.replace(" ", "_")

def normalize_wiki_link(href):
    """Return a clean absolute URL if `href` points to a real DramaWiki content
    page; otherwise None (red links, edit links, other sites, File:/Category: pages)."""
    url = urljoin(WIKI_BASE + "/", href)
    p = urlparse(url)
    if p.netloc.lower() != "wiki.d-addicts.com":
        return None
    if p.path.startswith("/index.php") or "redlink" in p.query or "action=" in p.query:
        return None
    name = unquote(p.path.lstrip("/"))
    if not name or ":" in name:
        return None
    return f"{WIKI_BASE}{p.path}"

def extract_ost_links(soup):
    """Links next to an 'Original Soundtrack:' label. One per season, typically."""
    urls = []
    for label in soup.find_all(["b", "strong"]):
        if label.get_text(strip=True).lower().startswith("original soundtrack"):
            container = label.find_parent("li") or label.parent
            for a in container.find_all("a"):
                url = normalize_wiki_link(a.get("href", ""))
                if url and url not in urls:
                    urls.append(url)
    return urls

def extract_main_cast_links(soup):
    """Return [{'name', 'url', 'season', 'note'}] for the 'Main Cast' list in
    every 'Cast' section of a show page."""
    results = []
    cast_headings = [
        h for h in soup.find_all(["h2", "h3", "h4"])
        if re.match(r"^\s*Cast\b", h.get_text(" ", strip=True))
    ]

    for season_no, heading in enumerate(cast_headings, start=1):
        # The Cast section = everything after its heading up to the next h2/h3.
        section = []
        for sib in heading.next_siblings:
            name = getattr(sib, "name", None)
            if name in ("h2", "h3"):
                break
            if name:
                section.append(sib)

        # Find the "Main Cast" label, then the first <ul> after it.
        marker = next(
            (i for i, el in enumerate(section)
             if el.name != "ul"
             and el.get_text(" ", strip=True).lower().rstrip(":") == "main cast"),
            None,
        )
        note = "" if marker is not None else "no 'Main Cast' label; used first list"
        start = 0 if marker is None else marker + 1
        ul = next((el for el in section[start:] if el.name == "ul"), None)
        if ul is None:
            continue

        count = 0
        for li in ul.find_all("li", recursive=False):      # top-level entries only
            # first link that belongs to THIS li, not to a nested sub-list
            a = next((a for a in li.find_all("a") if a.find_parent("ul") is ul), None)
            if a is None:
                continue
            url = normalize_wiki_link(a.get("href", ""))
            if url is None:                                  # red link / off-wiki
                continue
            results.append({"name": a.get_text(strip=True), "url": url,
                            "season": season_no, "note": note})
            count += 1
            if MAIN_CAST_LIMIT and count >= MAIN_CAST_LIMIT:
                break
    return results

<a id=5></a>
## 5. Downloading for every show

In [ ]:
def load_show_samples():
    """
    Load the music-centered and control samples.

    Every music-centered drama is retained.
    The first N control dramas are selected, where N is the
    number of music-centered dramas.

    The original ordering of the control dataset is preserved.
    """

    music_df = pd.read_csv(MUSIC_CSV)
    control_df = pd.read_csv(CONTROL_CSV)

    # Check that the expected title column exists.
    for name, frame in [
        ("music", music_df),
        ("control", control_df)
    ]:
        if TITLE_COLUMN not in frame.columns:
            raise ValueError(
                f"{TITLE_COLUMN} not found in {name} dataset."
            )

    # Remove rows without a title.
    music_df = music_df.dropna(subset=[TITLE_COLUMN]).copy()
    control_df = control_df.dropna(subset=[TITLE_COLUMN]).copy()

    # Remove duplicate titles within each group.
    music_df = music_df.drop_duplicates(subset=[TITLE_COLUMN])
    control_df = control_df.drop_duplicates(subset=[TITLE_COLUMN])

    # Determine the required control sample size.
    n_music = len(music_df)

    if CONTROL_SAMPLE_SIZE is not None:
        n_control = min(CONTROL_SAMPLE_SIZE, n_music)
    else:
        n_control = n_music

    # Preserve the existing order of the control dataset.
    control_df = control_df.head(n_control).copy()

    # Add group labels.
    music_df["group"] = "music"
    control_df["group"] = "control"

    # Combine the datasets.
    combined = pd.concat(
        [music_df, control_df],
        ignore_index=True
    )

    # Create a consistent title column
    combined["title"] = combined[TITLE_COLUMN].astype(str).str.strip()

    # Apply known manual URL overrides.
    combined["wiki_page"] = combined["title"].map(WIKI_PAGE_OVERRIDES)

    # Generate provisional page names for titles without overrides.
    # Remove trailing release years before constructing the Wiki URL.
    combined["wiki_page"] = combined.apply(
        lambda row: (
            row["wiki_page"]
            if pd.notna(row["wiki_page"])
            else re.sub(r"\s*\(?\b(?:19|20)\d{2}\)?$", "", row["title"]).replace(" ", "_")
        ),
        axis=1
    )

    print(f"Music-centered dramas: {len(music_df)}")
    print(f"Control dramas: {len(control_df)}")
    print(f"Total dramas: {len(combined)}")

    assert len(music_df) == len(control_df), ("The two samples are not equal in size.")
    return combined.to_dict("records")

def download_everything(shows):
    manifest = []

    def log(show_title, group, kind, extra, rec):
        manifest.append({
            "show": show_title,
            "group": group,
            "kind": kind,
            **extra,
            **rec
        })

        print(
            f"    {kind:6} {rec['status']!s:7} "
            f"{rec['url']}  {rec['note']}"
        )

    for show in shows:
        title = show["title"]
        group = show["group"]
        page = show["wiki_page"]

        show_url = wiki_url(page)

        print(f"\\n=== [{group}] {title} ===")

        # 1) Download the show page
        rec = fetch_page(show_url, local_path("shows", show_url))
        log(title, group, "show", {}, rec)

        if not Path(rec["path"]).exists():
            print(
                "    -> couldn't get the show page, "
                "skipping its OST/cast pages"
            )
            continue

        soup = BeautifulSoup(Path(rec["path"]).read_bytes(), "html.parser")

        # 2) Download OST pages
        ost_urls = extract_ost_links(soup)

        if not ost_urls: print("    (no Original Soundtrack link found)")

        for ost_url in ost_urls:
            rec = fetch_page(ost_url, local_path("osts", ost_url))
            log(title, group, "ost", {}, rec)

        # 3) Download main-cast actor pages
        cast = extract_main_cast_links(soup)

        print(f"    found {len(cast)} main-cast links")

        for member in cast:
            rec = fetch_page(member["url"], local_path("actors", member["url"]))

            log(
                title,
                group,
                "actor",
                {
                    "actor_name": member["name"],
                    "season": member["season"],
                    "cast_note": member["note"]
                },
                rec
            )

    return manifest

In [ ]:
def create_drama_summary(manifest):
    """
    Create one summary row per drama, showing which Wiki pages
    were successfully retrieved.
    """

    m = pd.DataFrame(manifest)

    if m.empty:
        print("Manifest is empty; no summary generated.")
        return pd.DataFrame()

    # A page is successful if it was newly downloaded or cached.
    m["success"] = (
        m["status"].astype(str).str.lower().isin(["200", "cached"])
    )

    summary_rows = []

    for (show, group), drama in m.groupby(["show", "group"], sort=False):
        show_rows = drama[drama["kind"] == "show"]
        ost_rows = drama[drama["kind"] == "ost"]
        actor_rows = drama[drama["kind"] == "actor"]

        # Successfully retrieved actor pages.
        successful_actors = actor_rows[actor_rows["success"]]

        # All attempted actor names.
        attempted_actors = (actor_rows["actor_name"].dropna().astype(str).drop_duplicates().tolist())

        # Successfully retrieved actor names.
        found_actors = (successful_actors["actor_name"].dropna().astype(str).drop_duplicates().tolist())

        # Actors whose pages could not be retrieved.
        missing_actors = [
            actor for actor in attempted_actors
            if actor not in found_actors
        ]

        # Retrieve the show page status.
        show_found = (
            show_rows["success"].any()
            if not show_rows.empty else False
        )

        show_status = (
            show_rows.iloc[0]["status"]
            if not show_rows.empty else "not_attempted"
        )

        # OST information.
        ost_found = ost_rows["success"].any()
        ost_attempted = len(ost_rows)

        # Actor information.
        summary_rows.append({
            "show": show,
            "group": group,

            "show_page_found": show_found,
            "show_page_status": show_status,

            "ost_pages_attempted": ost_attempted,
            "ost_pages_found": int(ost_rows["success"].sum()),
            "ost_page_found": ost_found,

            "actors_attempted": len(attempted_actors),
            "actor_pages_found": len(found_actors),
            "actor_pages_missing": len(missing_actors),

            "found_actors": "; ".join(found_actors),
            "missing_actors": "; ".join(missing_actors),
        })

    return pd.DataFrame(summary_rows)

<a id=6></a>
## 6. Run it

In [ ]:
# Reset counters for this run.
FETCH_COUNTS = {
    "cached": 0,
    "downloaded": 0,
    "failed": 0,
}

manifest = []

try:
    shows = load_show_samples()
    selected_df = pd.DataFrame(shows)

    selected_df.to_csv(
        RAW_DIR / "selected_drama_sample.csv",
        index=False
    )

    print(selected_df.groupby("group").size())
    manifest = download_everything(shows)

finally:
    close_driver()
    if manifest:
        # Save detailed manifest.
        manifest_df = pd.DataFrame(manifest)

        manifest_df.to_csv(MANIFEST_PATH, index=False)
        print(f"\nManifest saved to {MANIFEST_PATH}")

        # Generate and save one-row-per-drama summary.
        summary_df = create_drama_summary(manifest)

        summary_path = "../dataset2_mediawiki/drama_summary.csv"
        summary_df.to_csv(summary_path, index=False)

        print(f"Drama summary saved to {summary_path}")

        display(summary_df.head(10)) # Display summary.

    print("\nFetch summary:")
    for key, value in FETCH_COUNTS.items():
        print(f"  {key}: {value}")
        
    print("\nCache summary:")
    print(f"  Successful HTML files cached: {FETCH_COUNTS['cached']}")
    print(f"  Previously confirmed 404s skipped: {sum(
        1 for row in manifest
        if row.get('status') == 'cached_404'
    )}")
    print(f"  New pages downloaded: {FETCH_COUNTS['downloaded']}")
    print(f"  Failed requests: {FETCH_COUNTS['failed']}")
    print(f"  Total URLs in persistent 404 cache: {len(NOT_FOUND_CACHE)}")

Music-centered dramas: 110
Control dramas: 110
Total dramas: 220
group
control    110
music      110
dtype: int64
\n=== [music] Sing My Crush (2023) ===
    show   404     https://wiki.d-addicts.com/Sing_My_Crush  Page does not exist; added to 404 cache
    -> couldn't get the show page, skipping its OST/cast pages
\n=== [music] Sound Candy (2023) ===
    show   cached  https://wiki.d-addicts.com/Sound_Candy  
    ost    cached  https://wiki.d-addicts.com/Sound_Candy_OST  
    found 3 main-cast links
    actor  cached  https://wiki.d-addicts.com/Choi_Yoo_Jung_(1999)  
    actor  cached  https://wiki.d-addicts.com/JR_(1995)  
    actor  cached  https://wiki.d-addicts.com/Baek_Seo_Hoo  
\n=== [music] Dream Knight (2015) ===
    show   cached  https://wiki.d-addicts.com/Dream_Knight  
    ost    cached  https://wiki.d-addicts.com/Dream_Knight_OST  
    found 6 main-cast links
    actor  cached  https://wiki.d-addicts.com/Song_Ha_Yoon  
    actor  cached  https://wiki.d-addicts.com/JB  
  

,show,group,show_page_found,show_page_status,ost_pages_attempted,ost_pages_found,ost_page_found,actors_attempted,actor_pages_found,actor_pages_missing,found_actors,missing_actors
0,Sing My Crush (2023),music,False,404,0,0,False,0,0,0,,
1,Sound Candy (2023),music,True,cached,1,1,True,3,3,0,Choi Yoo Jung; JR; Baek Seo Hoo,
2,Dream Knight (2015),music,True,cached,1,1,True,6,6,0,Song Ha Yoon; JB; Jr.; Choi Young Jae; Park Ji...,
3,Idol: The Coup (2021),music,True,cached,0,0,False,3,3,0,Hani; Kwak Shi Yang; Kim Min Gyu,
4,Maestra: Strings of Truth (2023),music,True,cached,0,0,False,3,3,0,Lee Young Ae; Lee Moo Saeng; Kim Young Jae,
5,Must You Go? (2021),music,False,404,0,0,False,0,0,0,,
6,Entertainer (2016),music,True,cached,1,1,True,4,4,0,Ji Sung; Hye Ri; Chae Jung Ahn; Kang Min Hyuk,
7,Spring of Youth (2025),music,True,cached,1,1,True,3,3,0,Ha Yoo Joon; Park Ji Hoo; Lee Seung Hyup,
8,The Miracle (2016),music,True,cached,1,1,True,1,1,0,Hong Yoon Hwa,
9,Miracle (2022),music,True,cached,0,0,False,11,11,0,Jang Yong; Park Won Sook; Jung Ki Sung; Sa Kan...,



Fetch summary:
  cached: 0
  downloaded: 0
  failed: 0

Cache summary:
  Successful HTML files cached: 0
  Previously confirmed 404s skipped: 0
  New pages downloaded: 0
  Failed requests: 0
  Total URLs in persistent 404 cache: 88


<a id=7></a>
## 7. Summary

In [45]:
m = pd.DataFrame(manifest)

if m.empty:
    print("Nothing in the manifest.")

else:
    # Show page coverage
    show_manifest = m[m["kind"] == "show"].copy()

    print("Show pages by group and outcome:")
    print(
        show_manifest.groupby(
            ["group", "status"]
        ).size().unstack(fill_value=0)
    )

    # Count successfully downloaded show pages.
    successful = show_manifest[
        show_manifest["status"].isin(["200", "cached", 200])
    ]

    print("\\nSuccessfully retrieved show pages:")
    print(successful.groupby("group").size())

    # Compare group sizes.
    coverage = successful.groupby("group")["show"].nunique()

    music_count = coverage.get("music", 0)
    control_count = coverage.get("control", 0)

    print(f"\\nMusic pages: {music_count}")
    print(f"Control pages: {control_count}")

    if music_count == control_count:
        print("The successfully retrieved samples are balanced.")
    else:
        print(
            "The successfully retrieved samples are NOT balanced."
        )

    # Show pages with problems.
    problems = m[
        (m["note"] != "")
        & (~m["status"].isin(["cached"]))
    ]

    print(f"\\n{len(problems)} pages with notes/problems:")

    if len(problems):
        display(
            problems[
                ["show", "group", "kind", "url", "status", "note"]
            ]
        )

    # Files on disk.
    print("\\nFiles on disk:")

    for sub in ("shows", "osts", "actors"):
        count = len(list((RAW_DIR / sub).glob("*.html")))
        print(f"  raw_html/{sub}/: {count} files")

Show pages by group and outcome:
status   200  404  cached
group                    
control    4   42      64
music      2   46      62
\nSuccessfully retrieved show pages:
group
control    68
music      64
dtype: int64
\nMusic pages: 64
Control pages: 68
The successfully retrieved samples are NOT balanced.
\n88 pages with notes/problems:


,show,group,kind,url,status,note
0,Sing My Crush (2023),music,show,https://wiki.d-addicts.com/Sing_My_Crush,404,Page does not exist; added to 404 cache
22,Must You Go? (2021),music,show,https://wiki.d-addicts.com/Must_You_Go%3F,404,Page does not exist; added to 404 cache
62,Her Bucket List (2021),music,show,https://wiki.d-addicts.com/Her_Bucket_List,404,Page does not exist; added to 404 cache
92,Nine Plus (2021),music,show,https://wiki.d-addicts.com/Nine_Plus,404,Page does not exist; added to 404 cache
93,K-Idol Invasion (2025),music,show,https://wiki.d-addicts.com/K-Idol_Invasion,404,Page does not exist; added to 404 cache
...,...,...,...,...,...,...
872,Secret Friend (2021),control,show,https://wiki.d-addicts.com/Secret_Friend,404,Page does not exist; added to 404 cache
876,My Dearest Part 2 (2023),control,show,https://wiki.d-addicts.com/My_Dearest_Part_2,404,Page does not exist; added to 404 cache
877,Reunion Counseling (2024),control,show,https://wiki.d-addicts.com/Reunion_Counseling,404,Page does not exist; added to 404 cache
878,The Story of Bi Hyeong: Enchanted Master of th...,control,show,https://wiki.d-addicts.com/The_Story_of_Bi_Hye...,404,Page does not exist; added to 404 cache


\nFiles on disk:
  raw_html/shows/: 155 files
  raw_html/osts/: 129 files
  raw_html/actors/: 581 files


In [46]:
show_manifest = pd.DataFrame(manifest)
show_manifest = show_manifest[
    show_manifest["kind"] == "show"
]

missing_shows = show_manifest[
    ~show_manifest["status"].isin(["200", "cached", 200])
]

missing_shows.to_csv(
    RAW_DIR / "missing_show_pages.csv",
    index=False
)

I'm exporting a couple of relevant subsets to CSVs.

 - dramas_with_show_pages : any dramas that were found on mediawiki.
 - dramas_with_show_ost_pages : any dramas that were found on mediawiki with OST pages as well
 - dramas_with_show_ost_actor_pages : any dramas that were found on mediawiki with OST pages and at least one actor page
 - drama_summary : the full set of 220 dramas and the finding results

 - final_drama_set : the final set of 47 dramas each, music and control

In [49]:
from pathlib import Path

OUTPUT_DIR = Path("../dataset2_mediawiki")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Subset 1: Show, OST, and actor pages all available
complete_subset = summary_df[
    (summary_df["show_page_found"]) &
    (summary_df["ost_pages_found"] >= 1) &
    (summary_df["actor_pages_found"] >= 1)
].copy()

# Subset 2: Show page available
show_subset = summary_df[
    summary_df["show_page_found"]
].copy()

ost_subset = summary_df[
    (summary_df["show_page_found"]) &
    (summary_df["ost_pages_found"] >= 1)
]
# Save
complete_subset.to_csv(
    OUTPUT_DIR / "dramas_with_show_ost_actor_pages.csv",
    index=False
)

show_subset.to_csv(
    OUTPUT_DIR / "dramas_with_show_pages.csv",
    index=False
)

ost_subset.to_csv(
    OUTPUT_DIR / "dramas_with_show_ost_pages.csv",
    index=False
)

print(f"Complete subset: {len(complete_subset)} dramas")
print(f"Show-page subset: {len(show_subset)} dramas")
print(f"Ost-page subset: {len(ost_subset)} dramas")

Complete subset: 98 dramas
Show-page subset: 132 dramas
Ost-page subset: 106 dramas
